<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# From unsupervised to semi-supervised

**[From unsupervised to semi-supervised](https://learning.nextia-ai.com/courses/anomaly-detection/m08/from-unsupervised-to-semi-supervised/)** · Anomaly Detection: Find What Is Unusual · Module 8, lesson 1 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** use the reviewers' answers ("real" or "not") to train a small classifier that re-orders the detector's alerts, and measure what it changes at the 50-alert budget.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 7 MB). No account. |
| **You should know** | Precision at the budget, from [The alert budget](https://learning.nextia-ai.com/courses/anomaly-detection/m07/the-alert-budget/); logistic regression, from the machine learning course. |
| **You do not need** | A local project. The setup below downloads everything. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m08/from-unsupervised-to-semi-supervised/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M08-L01-from-unsupervised-to-semi-supervised/re-ranker-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`payments.csv.gz`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py` and `features.py`, the course's shared code. It needs an internet connection and no account. It takes about 10 seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "payments.csv.gz": "c4e18c1bbaf422cfc99821f2a1b65cf0e8274f835c40ca757a37a531eeccd542",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py", "features.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The reviewers already know something

Every day, Sofia's team reviews 50 alerts and marks each one **real** (fraud) or **not**. The detector ignores these answers: it scores tomorrow's payments exactly as it scored today's. A new colleague would learn from them: "alerts with a new device at night were real; alerts of business customers at noon were not".

This is **semi-supervised** detection: an unsupervised detector finds the candidates, and a few labels teach a second model to put the real ones first. The second model is a **re-ranker**: it does not look at every payment, only at the detector's top candidates.

## 2. Candidates and reviews

Run the next cell. Isolation Forest scores the evaluation months; each day, its 200 highest scores are the candidates, and the top 50 are shown to the team.

> **Check.** You should see 18,200 candidates in 91 days.

In [ ]:
import compare, features
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
payments = pd.read_csv("data/payments.csv.gz", parse_dates=["ts"])
reference, evaluation = compare.split(payments)
behaviour = features.behaviour(payments)
scaler = StandardScaler().fit(behaviour.loc[reference.index])
forest = IsolationForest(n_estimators=200, random_state=0, n_jobs=-1).fit(scaler.transform(behaviour.loc[reference.index]))
df = evaluation[["payment_id", "ts", "amount"]].copy()
df["day"] = df.ts.dt.floor("D")
df["iso"] = -forest.score_samples(scaler.transform(behaviour.loc[evaluation.index]))
for col in behaviour.columns:
    df[col] = behaviour.loc[evaluation.index, col].to_numpy()
df = df.sort_values(["day", "iso"], ascending=[True, False])
df["rank"] = df.groupby("day").cumcount()
candidates = df[df["rank"] < 200].copy()          # each day, the 200 highest scores
candidates["shown"] = candidates["rank"] < 50     # the team reviews the top 50
print(f"{len(candidates):,} candidates in {candidates.day.nunique()} days")

Run the next cell. It plays the reviewers for the first four weeks (September 2025): every alert they saw gets its answer from the answer key. Real reviewers make mistakes; here we assume they do not.

> **Check.** You should see 1,400 reviewed alerts, 708 real (50.6%).

In [ ]:
# The reviewers' answers. We play the reviewers with the answer key (an assumption: real reviewers make mistakes).
real = set(compare.planted_payments().payment_id)
candidates["label"] = candidates.payment_id.isin(real)
train_weeks = candidates.day <= "2025-09-28"                 # four weeks of reviews
labels = candidates[train_weeks & candidates.shown]          # only what the team saw has a label
print(f"{len(labels):,} reviewed alerts in 4 weeks, {labels.label.sum():,} real ({labels.label.mean():.1%})")

## 3. Train the re-ranker

The re-ranker is a logistic regression on the Isolation Forest score and the eight behaviour features, trained on the reviewed alerts only. Run the next cell: it prints the weight of each feature (scaled, so you can compare them).

> **Check.** The three largest weights (in size) are iso, new_device, is_business.

In [ ]:
FEATS = ["iso"] + list(behaviour.columns)
re_scaler = StandardScaler().fit(labels[FEATS])
re_ranker = LogisticRegression(max_iter=2000).fit(re_scaler.transform(labels[FEATS]), labels.label)
coef = pd.Series(re_ranker.coef_[0], index=FEATS).round(2).sort_values()
coef

> **Predict.** In the following weeks (29 September to 30 November), the re-ranker re-orders each day's 200 candidates, and the team still reviews 50. Will precision at the budget go up, and by how much?

In [ ]:
test = candidates[~train_weeks].copy()
test["re_rank"] = re_ranker.predict_proba(re_scaler.transform(test[FEATS]))[:, 1]

def precision_at_50(frame, column):
    top = frame.sort_values(["day", column], ascending=[True, False]).groupby("day").head(50)
    return round(float(top.label.mean()), 3), round(float(top.label.sum() / max(frame.label.sum(), 1)), 3)

before, after = precision_at_50(test, "iso"), precision_at_50(test, "re_rank")
print(f"test weeks: Isolation Forest alone {before[0]} precision, {before[1]} of the real candidates found")
print(f"            re-ranked             {after[0]} precision, {after[1]} of the real candidates found")

> **Check.** You should see 0.498 for Isolation Forest alone and 0.514 re-ranked.

The change is small: 0.498 → 0.514. Isolation Forest on good features already puts most planted frauds in the top 50, and on many days there are fewer than 50 planted payments, so even a perfect order cannot fill the budget with real ones. A re-ranker helps most when the detector's top alerts mix many false ones with the real ones.

## 4. A trap to keep in mind

The re-ranker learned only from the alerts that the team **saw**: the top 50 of a detector. It never sees a fraud that the detector ranked 120th. The next lesson, *The feedback loop and its traps*, measures this selection bias and the step that reduces it.

## 5. Check yourself

**Your turn 1.** What share of the reviewed alerts of the four weeks are real? Write it as `real_share_in_labels`, rounded to 3 decimals.

In [ ]:
real_share_in_labels = ...   # your code

In [ ]:
expect('real_share_in_labels', real_share_in_labels, 0.506, tolerance=0.001)

**Your turn 2.** Train the re-ranker on **two** weeks of reviews (up to 2025-09-14) instead of four, and compute its precision at the budget on the same test weeks. Write it as `precision_two_weeks`.

In [ ]:
precision_two_weeks = ...   # your code

In [ ]:
expect_hash('precision_two_weeks', precision_two_weeks, 'b99471154c8b8586')

In the solution: 0.511 with two weeks of reviews, 0.514 with four.

## Recap

- Reviewers' answers are labels. A small classifier on top of the anomaly score (a re-ranker) can use them to put the real alerts first.
- Measure it at the budget, on later weeks than it learned from: here 0.498 → 0.514.
- The labels come only from what the detector showed: that is the trap of the next lesson.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [The feedback loop and its traps](https://learning.nextia-ai.com/courses/anomaly-detection/m08/the-feedback-loop-and-its-traps/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m08/from-unsupervised-to-semi-supervised/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.